# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Runtime → **Change runtime type → GPU**. This notebook clones the harness, installs the official LeWM stack, reproduces LeWM (Phase 1), then trains/evals the nGPT-JEPA variants. The baseline is **official LeWM, unchanged**.

See `official_lewm_reproduction.md`, `RUN_MATRIX.md`, `report.md`.

## 0. GPU + clone

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime type to GPU'

In [ ]:
!git clone -b claude/upbeat-babbage-kbmgsr https://github.com/turtlenottortoise/spinangle.git 2>/dev/null || (cd spinangle && git pull)
%cd spinangle
!git log --oneline -1

## 1. Install
Official LeWM stack (`stable-worldmodel[train,env]` → stable-pretraining, mujoco, hydra) + matplotlib. Takes a few minutes.

In [ ]:
import os
os.environ['STABLEWM_HOME'] = '/content/stable-wm'
os.environ['MUJOCO_GL'] = 'egl'
!mkdir -p $STABLEWM_HOME
!pip -q install 'stable-worldmodel[train,env]' matplotlib einops

## 2. CPU smoke test (no data needed)
Validates every variant: forward/backward, unit-norm invariants, planner path, and that the official path matches the reference LeWM loss.

In [ ]:
!python smoke_test.py
!python metrics.py

## 3. Download official data + checkpoints
From the HuggingFace collection `quentinll/lewm`. Edit the dataset/ckpt names per benchmark. Datasets are HDF5; extract under `$STABLEWM_HOME`.

```bash
# example (Push-T): download + convert the HF checkpoint to the object ckpt
# eval.py expects — see UPSTREAM_README.md for the exact conversion snippet.
hf download quentinll/lewm-pusht --local-dir $STABLEWM_HOME/hf_pusht
```

In [ ]:
# TODO: fetch datasets (.h5) into $STABLEWM_HOME and the pretrained ckpts.
# Follow official_lewm_reproduction.md (Data + Phase 1a) and UPSTREAM_README.md.
!ls -la $STABLEWM_HOME

## 4. Phase 1 — reproduce official LeWM
**Gate:** do not move to variants until at least one benchmark reproduces.

In [ ]:
# 1a. pretrained-checkpoint eval (fastest)
!python eval.py --config-name=pusht.yaml policy=pusht/lewm
# 1b. short sanity training
!python train.py +experiment=official_lewm data=pusht trainer.max_epochs=2 wandb.enabled=false

In [ ]:
# log the reproduction result
!python scripts/log_run.py --variant official_lewm --benchmark pusht --phase 1 \
    --success 0.0 --plan_samples 300 --train_epochs 100 --ckpt_path pusht/lewm  # edit numbers

## 5. Phases 2-6 — train + eval variants
Same `train.py`/`eval.py`, same planner budget; only `+experiment=` changes.
Set `EPOCHS=100` for the matched budget (use a small value first to validate).

In [ ]:
VARIANTS = [
    'official_lewm', 'lewm_nosigreg', 'simple_spherical', 'fullish_residual',
    'gated_spherical', 'gated_spherical_projector_sigreg',
    'gated_spherical_memory', 'gated_spherical_ssm',
]
DATA = 'pusht'      # pusht | dmc | tworoom | ogb
EPOCHS = 5          # bump to 100 for the matched-compute comparison
for v in VARIANTS:
    print('=== train', v, '===')
    !python train.py +experiment={v} data={DATA} trainer.max_epochs={EPOCHS} wandb.enabled=false

In [ ]:
# planning eval (LeWM eval.py, unchanged) — assumes ckpts at <benchmark>/<variant>
for v in VARIANTS:
    print('=== eval', v, '===')
    !python eval.py --config-name=pusht.yaml policy=pusht/{v} || echo 'set policy path'

## 6. Offline latent / retrieval / representation metrics

In [ ]:
for v in VARIANTS:
    sph = '' if v in ('official_lewm','lewm_nosigreg') else '--spherical'
    !python scripts/eval_latent_metrics.py --policy pusht/{v} --data pusht \
        --benchmark pusht --variant {v} {sph} --horizon 20 --num_batches 16 || true

## 7. Plots + report

In [ ]:
!python scripts/make_plots.py
from IPython.display import Image, display
for p in ['success_vs_steps','rollout_error_vs_horizon','retrieval_vs_steps',
          'rank_clumping','planning_budget_curve']:
    display(Image(f'plots/{p}.png'))

In [ ]:
# persist results back to the branch (optional)
!git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure auth to push'